In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/omarahmed05/target-rules/file.txt


In [2]:
%pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [11]:
from langchain_groq import ChatGroq
from pydantic import BaseModel, ConfigDict, Field
from typing import List

api_key = "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"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=api_key,
    temperature=0,
)

In [12]:
!pip install -U langchain-classic

In [13]:
from typing import List
from pydantic import BaseModel, Field, ConfigDict
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate


class CVSkills(BaseModel):
    model_config = ConfigDict(extra="forbid")

    skills: List[str] = Field(
        description="Distinct reusable skills explicitly supported by the CV."
    )


skills_parser = PydanticOutputParser(
    pydantic_object=CVSkills
)

skills_format_instructions = skills_parser.get_format_instructions()


extraction_prompt = """
You are an evidence-based CV skill extraction system for any occupation,
industry, or career level.

Extract up to 25 distinct, reusable skills that the CV explicitly supports.
Do not fill the limit: return fewer skills when that is what the evidence
supports.

A skill is a transferable capability, area of knowledge, method, tool,
technology, or professional competency that could reasonably appear in a
Skills section. This can include technical, scientific, operational,
business, creative, administrative, interpersonal, and other occupational
skills.

Review the entire CV. Skills may be listed directly or supported by
descriptions of training, courses, projects, achievements, and work
experience. A course or project can provide evidence for a skill, but its
title alone is not a skill.

Apply these rules:

1. Require evidence. Include a skill only if the CV names it or clearly
   shows the candidate used, applied, or learned it. Do not infer skills
   from a job title, project title, link, or general knowledge.
2. Extract reusable capabilities. Exclude project names, products,
   application features, one-time activities, deliverables, and outcomes.
3. Choose the right level of detail. Include meaningful named skills, but
   do not turn every component, step, feature, or vocabulary term into a
   separate skill. Combine overlapping details under a clear, broader skill
   when they describe parts of that skill.
4. Avoid duplication. Do not list a skill together with minor variations,
   repeated mentions, or sub-items that add no distinct capability.
5. Use concise, conventional names. Correct an obvious spelling or OCR error
   only when the intended skill is unambiguous. Do not guess unclear terms.
6. Include professional or interpersonal skills only when the CV explicitly
   identifies them or provides clear evidence. Do not infer them from
   participation or project completion.
7. Exclude personal information, natural languages, locations, institutions,
   degrees, and course titles. A skill taught or practiced in a course may
   still be included when the CV identifies that skill.

Before including each item, check:
- Is this a skill rather than a project detail or generic noun?
- Is it supported by this CV?
- Is it meaningfully distinct from the other selected skills?

Return exactly one valid JSON object with a single key, "skills", whose value
is an array of strings. Return no Markdown, explanation, or extra text.

Required structure:
{{
  "skills": ["skill name", "another skill"]
}}

Follow this schema:
{format_instructions}

CV:
{cv_text}
"""

skill_extraction_template = PromptTemplate(
    template=extraction_prompt,
    input_variables=["cv_text", "format_instructions"],
)

print(skills_format_instructions)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"additionalProperties": false, "properties": {"skills": {"description": "Distinct reusable skills explicitly supported by the CV.", "items": {"type": "string"}, "title": "Skills", "type": "array"}}, "required": ["skills"]}
```


In [14]:
import json
import re

def extract_json_block(text):
    text = text.strip()

    # First try valid JSON.
    try:
        data = json.loads(text)

        if isinstance(data, list):
            return json.dumps({"skills": data}, ensure_ascii=False)

        if isinstance(data, dict) and isinstance(data.get("skills"), list):
            return json.dumps(data, ensure_ascii=False)

    except json.JSONDecodeError:
        pass

    # Recover skill strings from a malformed JSON array.
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end > start:
        array_text = text[start + 1:end]

        skills = re.findall(
            r'"((?:\\.|[^"\\])*)"',
            array_text,
            flags=re.DOTALL
        )

        skills = [
            re.sub(r"\s+", " ", skill).strip()
            for skill in skills
            if skill.strip()
        ]

        if skills:
            return json.dumps({"skills": skills}, ensure_ascii=False)

    raise ValueError("Could not recover a skills list from the model response.")


cv_chain = skill_extraction_template | llm.with_structured_output(
    CVSkills,
    method="json_schema",
    strict=True,
)

def extract_cv_data(full_text):
    return cv_chain.invoke({
        "cv_text": full_text,
        "format_instructions": skills_format_instructions,
    })
   

In [15]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.6 MB/s eta 0:00:00:00:010:01


In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import re

def chunk_text(text, chunk_size=500, overlap=50):
    role_headers = list(re.finditer(
        r"(?m)^ROLE_ID:\s*[a-z0-9_]+\s*$",
        text,
    ))

    chunks = []

    for i, header in enumerate(role_headers):
        start = header.start()
        end = (
            role_headers[i + 1].start()
            if i + 1 < len(role_headers)
            else len(text)
        )
        chunks.append(text[start:end].strip())

    return chunks

def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]
    

In [17]:
import re
from pathlib import Path
from typing import List

from pydantic import BaseModel, ConfigDict, Field
from langchain_core.prompts import PromptTemplate


class TargetSkills(BaseModel):
    model_config = ConfigDict(extra="forbid")

    skills: List[str] = Field(
        description="Core and conditional skills stated in the retrieved target role profile."
    )


target_skills_prompt = PromptTemplate(
    template="""
Extract the core and conditional skills from this role profile.

Use only skills explicitly stated in the profile. Do not add skills from
general knowledge or include skills from another role.

Role profile:
{role_text}
""",
    input_variables=["role_text"],
)


target_skills_chain = target_skills_prompt | llm.with_structured_output(
    TargetSkills,
    method="json_schema",
    strict=True,
)


role_db_text = Path(
    "/kaggle/input/datasets/omarahmed05/target-rules/file.txt"
).read_text(encoding="utf-8")


chunks = chunk_text(role_db_text)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)


def get_target_skills(target_rule):
    print("get_target_skills() started")
    print("Target rule:", target_rule)

    question = (
        f"Find the complete role profile for '{target_rule}'. "
        "Retrieve its core and conditional skills."
    )

    top_chunks = search_index(
        question,
        model_embeddings,
        index,
        chunks,
        k=1,
    )

    if not top_chunks:
        raise ValueError(f"No role profile found for: {target_rule}")

    role_text = top_chunks[0]
    print("Retrieved role profile:")
    print(role_text)

    target_skills = target_skills_chain.invoke({
        "role_text": role_text,
    })

    print("Target skills:", target_skills.skills)
    print("get_target_skills() finished")

    return target_skills.skills

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
from typing import List

from pydantic import BaseModel, ConfigDict, Field
from langchain_core.prompts import PromptTemplate


class SkillComparison(BaseModel):
    model_config = ConfigDict(extra="forbid")

    matched_skills: List[str] = Field(
        description="Target skills clearly supported by the CV."
    )
    needed_skills: List[str] = Field(
        description=(
            "Target skills not clearly supported by the CV, including "
            "partially supported skills."
        )
    )


comparison_prompt_template = PromptTemplate(
    template="""
Compare the target skills against the candidate's CV conservatively.

Use the full CV as the evidence source. The CV skills JSON is a helpful
summary and may omit skills demonstrated elsewhere in the CV.

A target skill is matched only when the CV explicitly names it or clearly
shows its use or learning. Recognize clear synonyms, but do not infer skills
from general knowledge or unrelated project context.

Examples:
- A GitHub link does not prove Git.
- Java or C++ does not prove SQL, databases, APIs, cloud, or frameworks.
- Testing does not prove debugging.
- Digital design does not prove software design.
- A related skill is not automatically equivalent.

Put every unsupported or only partially supported target skill in
needed_skills. Include every target skill exactly once across the two lists.

CV skills JSON:
{cv_skills_json}

Full CV:
{cv_text}

Target skills JSON:
{target_skills_json}
""",
    input_variables=[
        "cv_skills_json",
        "cv_text",
        "target_skills_json",
    ],
)


comparison_chain = comparison_prompt_template | llm.with_structured_output(
    SkillComparison,
    method="json_schema",
    strict=True,
)


def get_needed_skills(
    cv_skills_json: str,
    target_skills_json: str,
    cv_text: str,
) -> List[str]:
    print("get_needed_skills() started")

    comparison = comparison_chain.invoke({
        "cv_skills_json": cv_skills_json,
        "cv_text": cv_text,
        "target_skills_json": target_skills_json,
    })

    print("Comparison returned:")
    print(comparison.model_dump_json(indent=2))
    print("Needed skills:")
    print(comparison.needed_skills)
    print("get_needed_skills() finished")

    return comparison.needed_skills

In [19]:
NGROK_TOKEN = "3K9TIXBx5IB11ViCyCfiz99ChtJ_334XFt6WQBPjjojmqLx9K"
API_KEY = "nothing111"

In [20]:
!pip install fastapi uvicorn pyngrok

In [22]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn
import threading
import time
import socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    print("===== REQUEST RECEIVED =====")

    try:
        data = await req.json()

        full_text = data.get("full_text", "")
        target_role = data.get("target_role", "")

        if not full_text:
            raise HTTPException(
                status_code=400,
                detail="full_text is required",
            )

        if not target_role:
            raise HTTPException(
                status_code=400,
                detail="target_role is required",
            )

        # Extract skills from the CV.
        cv_data = extract_cv_data(full_text)

        # Retrieve skills for the requested target role.
        target_skills = get_target_skills(target_role)

        # Compare the two JSON skill lists.
        needed_skills = get_needed_skills(
            cv_data.model_dump_json(),
            json.dumps({"skills": target_skills}),
            full_text,
        )

        return {
            "response": {
                "cv_skills": cv_data.skills,
                "target_skills": target_skills,
                "needed_skills": needed_skills,
            }
        }

    except HTTPException:
        raise

    except Exception as e:
        import traceback

        print("===== EXCEPTION =====")
        traceback.print_exc()

        raise HTTPException(
            status_code=500,
            detail=str(e),
        )
        

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port


port = free_port()

conf.get_default().auth_token = NGROK_TOKEN

ngrok.kill()

public_url = ngrok.connect(port).public_url

print("Your public URL:", public_url)


def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


threading.Thread(
    target=run,
    daemon=True
).start()

time.sleep(1)

print(f"API endpoint: {public_url}/generate")

Your public URL: https://economic-smith-gurgling.ngrok-free.dev


INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:45963 (Press CTRL+C to quit)


API endpoint: https://economic-smith-gurgling.ngrok-free.dev/generate
===== REQUEST RECEIVED =====
get_target_skills() started
Target rule: Software engineer
Retrieved role profile:
ROLE_ID: software_engineer
TARGET_ROLE: Software Engineer / Software Developer
CORE_SKILLS: Programming in a job-relevant language; data structures and algorithms; software design; debugging; testing; version control (Git); requirements analysis; maintainable code; collaboration.
STACK_SPECIFIC: SQL and databases; APIs; cloud; frontend or backend frameworks; operating systems and networking, depending on the job.
SOURCE: O*NET Software Developers, 15-1252.00 — https://www.onetonline.org/link/summary/15-1252.00
Target skills: ['Programming in a job-relevant language', 'data structures and algorithms', 'software design', 'debugging', 'testing', 'version control (Git)', 'requirements analysis', 'maintainable code', 'collaboration', 'SQL and databases', 'APIs', 'cloud', 'frontend or backend frameworks', 'operat